# Support Vector Machine (SVM)
Support Vector Machines (SVM) zijn krachtige en veelzijdige algoritmen voor zowel classificatie- als regressietaken. Het belangrijkste idee achter SVM is het vinden van een hypervlak dat de gegevenspunten van verschillende klassen in de feature-ruimte scheidt met de grootste marge. Dit hypervlak wordt bepaald door de zogenaamde "support vectors", de gegevenspunten die het dichtst bij het hypervlak liggen. SVM kan ook worden uitgebreid naar niet-lineaire classificatie door gebruik te maken van kernel-tricks, die de gegevens in een hogere dimensionale ruimte projecteren waar een lineaire scheiding mogelijk is. SVM is bijzonder effectief bij hoge-dimensionale datasets en wordt vaak gebruikt in toepassingen zoals tekstclassificatie en beeldherkenning.

# Comprehensive Support Vector Machine Analysis
This analysis evaluates SVM performance across multiple configurations:
- **Kernel Types**: Linear, RBF, Polynomial, Sigmoid kernels
- **Feature Representations**: Original features, PCA-reduced, LDA-reduced
- **Hyperparameter Optimization**: GridSearchCV for optimal parameters
- **SVM-Specific Analysis**: Support vectors, decision boundaries, kernel performance
- **Interpretability**: Feature importance through kernel analysis and support vector examination

In [6]:
%%time

# Essential imports for comprehensive SVM analysis
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import h5py
import warnings
from pathlib import Path
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.pipeline import Pipeline
from sklearn.base import clone
import time
from collections import defaultdict
import os
import re
import gc
from scipy import sparse
from scipy.sparse.linalg import spsolve
from scipy.signal import savgol_filter
from scipy.interpolate import interp1d
from scipy.ndimage import median_filter
import time

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# Set style for better visualizations
plt.style.use('default')
sns.set_palette("husl")

print("=== COMPREHENSIVE SVM ANALYSIS SETUP ===")
print("Libraries imported successfully!")
print("Available SVM kernels: Linear, RBF, Polynomial, Sigmoid")
print("Feature preprocessing: StandardScaler, RobustScaler, MinMaxScaler")
print("Dimensionality reduction: PCA, LDA")
print("Analysis ready to begin!")

=== COMPREHENSIVE SVM ANALYSIS SETUP ===
Libraries imported successfully!
Available SVM kernels: Linear, RBF, Polynomial, Sigmoid
Feature preprocessing: StandardScaler, RobustScaler, MinMaxScaler
Dimensionality reduction: PCA, LDA
Analysis ready to begin!
CPU times: total: 0 ns
Wall time: 1.56 ms


In [7]:
def extract_tire_number(filename):
    """Extract tire number from filename"""
    match = re.search(r'_B(\d+)_', filename)
    if match:
        return int(match.group(1))
    else:
        match = re.search(r'nr-(\d+)', filename)
        if match:
            return int(match.group(1))
    return None

def process_file_generator(data_directory, max_files=None):
    """
    Generator that yields individual measurements one at a time
    More memory efficient than storing all in memory
    """
    all_files = [f for f in os.listdir(data_directory) if f.endswith('.h5')]
    if max_files:
        all_files = all_files[:max_files]
    
    total_measurements = 0
    wavelength_reference = None  # Store wavelength array once
    
    for i, filename in enumerate(all_files):
        if i % 10 == 0:
            print(f"Processing file {i+1}/{len(all_files)}: {filename}")
        
        try:
            filepath = os.path.join(data_directory, filename)
            tire_number = extract_tire_number(filename)
            
            if tire_number is None:
                continue
            
            # Determine origin from filename
            if "tread" in filename.lower():
                origin = 'tread'
            elif "innerliner" in filename.lower():
                origin = 'innerliner'
            elif "sidewall" in filename.lower():
                origin = 'sidewall'
            else:
                continue
            
            # Load HDF5 file efficiently
            with h5py.File(filepath, 'r') as h5:
                intensity = h5['intensity'][:]
                wavelength = h5['wavelength'][:]
                
                # Store wavelength reference only once
                if wavelength_reference is None:
                    wavelength_reference = wavelength.copy()
                
                # Process measurements
                if intensity.ndim == 2:
                    n_measurements, n_wavelengths = intensity.shape
                    
                    # Yield each measurement individually
                    for measurement_idx in range(n_measurements):
                        measurement_data = {
                            'tire_number': tire_number,
                            'origin': origin,
                            'measurement_id': measurement_idx,
                            'intensities': intensity[measurement_idx, :].astype(np.float32),  # Use float32 to save memory
                        }
                        yield measurement_data
                        total_measurements += 1
                else:
                    # Single measurement case
                    measurement_data = {
                        'tire_number': tire_number,
                        'origin': origin,
                        'measurement_id': 0,
                        'intensities': intensity.astype(np.float32),
                    }
                    yield measurement_data
                    total_measurements += 1
                    
        except Exception as e:
            print(f"  Error processing {filename}: {e}")
            continue
    
    print(f"Total measurements processed: {total_measurements:,}")

# Memory-efficient data loading
data_directory = '../data'
all_files = [f for f in os.listdir(data_directory) if f.endswith('.h5')]
print(f"Found {len(all_files)} HDF5 files")

# Use all files (remove max_files limit for full dataset)
print("Processing ALL files for maximum dataset size...")

# Initialize data containers with pre-allocation
print("Initializing memory-efficient data structures...")

# First pass: count total measurements and get wavelength info
measurement_count = 0
wavelength_info = None

print("First pass: Counting measurements...")
for measurement in process_file_generator(data_directory):
    measurement_count += 1
    if wavelength_info is None:
        n_wavelengths = len(measurement['intensities'])
        wavelength_info = n_wavelengths
    
    # Progress tracking
    if measurement_count % 10000 == 0:
        print(f"  Counted {measurement_count:,} measurements...")

print(f"Total measurements found: {measurement_count:,}")
print(f"Wavelengths per spectrum: {wavelength_info}")

# Pre-allocate numpy arrays for maximum memory efficiency
print("Pre-allocating memory-efficient arrays...")
X_data = np.empty((measurement_count, wavelength_info), dtype=np.float32)
y_data = np.empty(measurement_count, dtype='U10')  # String array for origins
metadata = np.empty((measurement_count, 2), dtype=np.int32)  # tire_number, measurement_id

# Second pass: fill the arrays
print("Second pass: Loading data into arrays...")
idx = 0
for measurement in process_file_generator(data_directory):
    X_data[idx] = measurement['intensities']
    y_data[idx] = measurement['origin']
    metadata[idx, 0] = measurement['tire_number']
    metadata[idx, 1] = measurement['measurement_id']
    
    idx += 1
    
    # Progress tracking and memory cleanup
    if idx % 10000 == 0:
        print(f"  Loaded {idx:,}/{measurement_count:,} measurements ({idx/measurement_count*100:.1f}%)")
        gc.collect()  # Force garbage collection

# Create DataFrame efficiently
print("Creating final DataFrame...")
wavelength_columns = [f'{wl:.3f}' for wl in np.linspace(200, 1000, wavelength_info)]  # Approximate wavelength range

# Create DataFrame with pre-allocated data
df_individual = pd.DataFrame(X_data, columns=wavelength_columns, dtype=np.float32)
df_individual['origin'] = y_data
df_individual['tire_number'] = metadata[:, 0]  
df_individual['measurement_id'] = metadata[:, 1]

# Clean up intermediate arrays
del X_data, y_data, metadata
gc.collect()

file_count = len([f for f in all_files if extract_tire_number(f) is not None])

print(f"\n" + "="*70)
print("MEMORY-EFFICIENT DATASET CREATED")
print("="*70)
print(f"Total measurements: {len(df_individual):,}")
print(f"Files processed: {file_count}")
print(f"Dataset shape: {df_individual.shape}")
print(f"Memory usage: {df_individual.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

# Analyze the dataset
print(f"\nDataset analysis:")
print(f"Unique tire numbers: {df_individual['tire_number'].nunique()}")
print(f"Origins distribution:")
print(df_individual['origin'].value_counts())

print(f"\nMeasurements per tire-origin combination:")
measurements_per_combo = df_individual.groupby(['tire_number', 'origin']).size()
print(f"Average: {measurements_per_combo.mean():.1f}")
print(f"Min: {measurements_per_combo.min()}")
print(f"Max: {measurements_per_combo.max()}")

# Identify wavelength columns (now they're numeric)
wavelength_columns = [col for col in df_individual.columns if col not in ['origin', 'tire_number', 'measurement_id']]
print(f"\nWavelength features: {len(wavelength_columns)}")

print(f"\nSample of first few rows:")
display_cols = ['tire_number', 'origin', 'measurement_id'] + wavelength_columns[:5]
print(df_individual[display_cols].head())

Found 316 HDF5 files
Processing ALL files for maximum dataset size...
Initializing memory-efficient data structures...
First pass: Counting measurements...
Processing file 1/316: 2024-11-05T11-26-41_nr-001_B1_tread_aided_10Hz_280A.h5
Processing file 11/316: 2024-11-05T11-51-23_nr-011_B4_innerliner_aided_10Hz_280A.h5
Processing file 21/316: 2024-11-05T12-02-34_nr-021_B7_sidewall_aided_10Hz_280A.h5
Processing file 31/316: 2024-11-05T12-13-30_nr-031_B22_tread_aided_10Hz_280A.h5
Processing file 21/316: 2024-11-05T12-02-34_nr-021_B7_sidewall_aided_10Hz_280A.h5
Processing file 31/316: 2024-11-05T12-13-30_nr-031_B22_tread_aided_10Hz_280A.h5
Processing file 41/316: 2024-11-05T12-25-42_nr-041_B33_innerliner_aided_10Hz_280A.h5
Processing file 51/316: 2024-11-05T12-36-31_nr-051_B28_sidewall_aided_10Hz_280A.h5
Processing file 41/316: 2024-11-05T12-25-42_nr-041_B33_innerliner_aided_10Hz_280A.h5
Processing file 51/316: 2024-11-05T12-36-31_nr-051_B28_sidewall_aided_10Hz_280A.h5
Processing file 61/316

In [8]:
# === PREPARE INDIVIDUAL MEASUREMENTS FOR MACHINE LEARNING ===
print("\n" + "="*70)
print("PREPARING INDIVIDUAL MEASUREMENTS FOR ML")
print("="*70)

# Check what columns actually exist in df_individual
print(f"Available columns in df_individual: {list(df_individual.columns)[:10]}...")  # Show first 10 columns
print(f"Total columns: {len(df_individual.columns)}")

# Separate features and target from individual measurements
# Only include columns that actually exist
existing_meta_columns = ['tire_number', 'origin', 'measurement_id']
meta_columns = [col for col in existing_meta_columns if col in df_individual.columns]

# Find wavelength columns - they are numeric column names (not prefixed with 'wl_')
numeric_columns = [col for col in df_individual.columns if isinstance(col, (int, float)) or 
                  (isinstance(col, str) and col.replace('.', '').replace('-', '').isdigit())]
wavelength_columns = [col for col in numeric_columns if col not in meta_columns]

print(f"Meta columns found: {meta_columns}")
print(f"Wavelength features found: {len(wavelength_columns)}")

# Create ML-ready dataset
X_individual = df_individual[wavelength_columns].copy()
y_individual = df_individual['origin'].copy()

# Column mapping is not needed since columns are already numeric wavelength values
column_mapping = {}

X_individual = X_individual.rename(columns=column_mapping)

print(f"Feature matrix shape: {X_individual.shape}")
print(f"Target distribution:\n{y_individual.value_counts()}")

# Handle any missing or infinite values
print(f"\nData quality check:")
missing_count = X_individual.isnull().sum().sum()
print(f"Missing values: {missing_count}")

if missing_count > 0:
    print("Filling missing values with column means...")
    X_individual = X_individual.fillna(X_individual.mean())

# Check for infinite values
inf_count = np.isinf(X_individual.values).sum()
print(f"Infinite values: {inf_count}")

if inf_count > 0:
    print("Replacing infinite values with column means...")
    X_individual = X_individual.replace([np.inf, -np.inf], np.nan).fillna(X_individual.mean())

# Add metadata for reference
df_individual_ml = df_individual[meta_columns].copy()
df_individual_ml = pd.concat([df_individual_ml, X_individual], axis=1)
df_individual_ml['origin'] = y_individual

print(f"\nFinal ML dataset shape: {df_individual_ml.shape}")
print(f"Samples available: {len(df_individual_ml):,}")

# Split the data with stratification to ensure balanced train/test sets
from sklearn.model_selection import train_test_split

X_train_ind, X_test_ind, y_train_ind, y_test_ind = train_test_split(
    X_individual, y_individual, 
    test_size=0.2, 
    random_state=42, 
    stratify=y_individual
)

print(f"\nTrain/Test split:")
print(f"Training set: {X_train_ind.shape}")
print(f"Test set: {X_test_ind.shape}")

print(f"\nTraining set distribution:")
print(y_train_ind.value_counts())

print(f"\nTest set distribution:")
print(y_test_ind.value_counts())

print(f"\n✅ INDIVIDUAL MEASUREMENTS DATASET READY FOR ML!")
print(f"   • {len(df_individual_ml):,} individual spectral measurements")
print(f"   • {len(wavelength_columns)} wavelength features")  


PREPARING INDIVIDUAL MEASUREMENTS FOR ML
Available columns in df_individual: ['200.000', '200.098', '200.195', '200.293', '200.391', '200.489', '200.586', '200.684', '200.782', '200.879']...
Total columns: 8191
Meta columns found: ['tire_number', 'origin', 'measurement_id']
Wavelength features found: 8188
Feature matrix shape: (51453, 8188)
Target distribution:
origin
tread         17936
innerliner    17260
sidewall      16257
Name: count, dtype: int64

Data quality check:
Feature matrix shape: (51453, 8188)
Target distribution:
origin
tread         17936
innerliner    17260
sidewall      16257
Name: count, dtype: int64

Data quality check:
Missing values: 0
Missing values: 0
Infinite values: 0
Infinite values: 0

Final ML dataset shape: (51453, 8191)
Samples available: 51,453

Final ML dataset shape: (51453, 8191)
Samples available: 51,453

Train/Test split:
Training set: (41162, 8188)
Test set: (10291, 8188)

Training set distribution:
origin
tread         14349
innerliner    13808


In [ ]:
def baseline_als(y, lam=1e4, p=0.01, niter=10):
    """
    Asymmetric Least Squares baseline correction
    
    Parameters:
    y: signal
    lam: smoothness parameter (larger = smoother baseline)
    p: asymmetry parameter (0 < p < 1, smaller = more asymmetric)
    niter: number of iterations
    """
    L = len(y)
    D = sparse.diags([1,-2,1],[0,-1,-2], shape=(L,L-2))
    w = np.ones(L)
    
    for i in range(niter):
        W = sparse.spdiags(w, 0, L, L)
        Z = W + lam * D.dot(D.transpose())
        z = spsolve(Z, w*y)
        w = p * (y > z) + (1-p) * (y < z)
    
    return z

def baseline_4s_peak_filling(y, window_length=51, polyorder=3, iterations=4, 
                            threshold_factor=0.1, fill_factor=0.8):
    """
    4S Peak Filling baseline correction algorithm
    
    This algorithm iteratively identifies and fills peaks to estimate the baseline:
    1. Smooth the spectrum
    2. Identify peaks (points above smoothed baseline)
    3. Fill peaks by interpolation or median filtering
    4. Repeat until convergence
    """
    
    # Ensure window_length is odd and valid
    if window_length % 2 == 0:
        window_length += 1
    window_length = min(window_length, len(y)//4*2 + 1)  # Ensure reasonable size
    
    # Start with a copy of the original signal
    baseline = y.copy().astype(float)
    
    for iteration in range(iterations):
        # Step 1: Apply Savitzky-Golay smoothing
        try:
            smoothed = savgol_filter(baseline, window_length, polyorder)
        except ValueError:
            # Fallback to median filter
            smoothed = median_filter(baseline, size=min(window_length//2, 5))
        
        # Step 2: Identify peaks (points significantly above smoothed baseline)
        threshold = np.std(baseline - smoothed) * threshold_factor
        peak_mask = (baseline - smoothed) > threshold
        
        if np.sum(peak_mask) == 0:
            break
        
        # Step 3: Fill peaks
        baseline[peak_mask] = smoothed[peak_mask] * fill_factor + baseline[peak_mask] * (1 - fill_factor)
        
        # Additional interpolation for isolated peaks
        peak_indices = np.where(peak_mask)[0]
        if len(peak_indices) > 0:
            # Group consecutive peak indices
            peak_groups = []
            current_group = [peak_indices[0]]
            
            for i in range(1, len(peak_indices)):
                if peak_indices[i] - peak_indices[i-1] <= 2:  # Close peaks
                    current_group.append(peak_indices[i])
                else:
                    peak_groups.append(current_group)
                    current_group = [peak_indices[i]]
            peak_groups.append(current_group)
            
            # Interpolate across peak groups
            for group in peak_groups:
                if len(group) > 1:  # Only interpolate for groups of peaks
                    start_idx = max(0, group[0] - 2)
                    end_idx = min(len(baseline), group[-1] + 3)
                    
                    # Create interpolation points (excluding the peak region)
                    x_interp = np.concatenate([[start_idx], [end_idx]])
                    y_interp = np.concatenate([[baseline[start_idx]], [baseline[end_idx]]])
                    
                    if len(x_interp) >= 2 and start_idx < end_idx:
                        f_interp = interp1d(x_interp, y_interp, kind='linear', 
                                          fill_value='extrapolate')
                        baseline[group[0]:group[-1]+1] = f_interp(np.arange(group[0], group[-1]+1))
    
    return baseline

def baseline_hybrid_4s_als(y, window_length=51, polyorder=3, als_lam=1e4, als_p=0.01):
    """
    Hybrid baseline correction: 4S Peak Filling followed by AsLS refinement
    """
    
    # Step 1: 4S Peak Filling for initial baseline estimation
    baseline_4s = baseline_4s_peak_filling(y, window_length, polyorder, iterations=2)
    
    # Step 2: Apply AsLS to the 4S result for fine-tuning
    baseline_final = baseline_als(baseline_4s, lam=als_lam, p=als_p)
    
    return baseline_final

def apply_baseline_correction_batch(spectra_matrix, method_func, batch_size=100, **kwargs):
    """
    Apply baseline correction to multiple spectra efficiently in batches
    """
    n_spectra, n_wavelengths = spectra_matrix.shape
    corrected_spectra = np.zeros_like(spectra_matrix, dtype=np.float32)
    
    print(f"Applying baseline correction to {n_spectra} spectra in batches of {batch_size}...")
    
    for start_idx in range(0, n_spectra, batch_size):
        end_idx = min(start_idx + batch_size, n_spectra)
        batch_size_actual = end_idx - start_idx
        
        print(f"  Processing batch {start_idx//batch_size + 1}/{(n_spectra-1)//batch_size + 1} "
              f"(spectra {start_idx+1}-{end_idx})")
        
        for i in range(start_idx, end_idx):
            try:
                spectrum = spectra_matrix[i].astype(float)
                baseline = method_func(spectrum, **kwargs)
                corrected_spectra[i] = spectrum - baseline
                
            except Exception as e:
                print(f"    Warning: Failed to correct spectrum {i+1}: {e}")
                corrected_spectra[i] = spectra_matrix[i]  # Keep original on failure
    
    return corrected_spectra

print("Baseline correction functions implemented successfully!")

# Apply Hybrid 4S+AsLS baseline correction to training and test data
print(f"\nApplying Hybrid 4S+AsLS baseline correction to LIBS data...")
print(f"Original training data shape: {X_train_ind.shape}")
print(f"Original test data shape: {X_test_ind.shape}")

start_time = time.time()

# Apply baseline correction to training data
print("Correcting training data...")
X_train_corrected = apply_baseline_correction_batch(
    X_train_ind.values, 
    baseline_hybrid_4s_als,
    batch_size=100,
    window_length=51, 
    polyorder=3, 
    als_lam=1e4, 
    als_p=0.01
)

# Apply baseline correction to test data  
print("Correcting test data...")
X_test_corrected = apply_baseline_correction_batch(
    X_test_ind.values,
    baseline_hybrid_4s_als,
    batch_size=100,
    window_length=51,
    polyorder=3,
    als_lam=1e4,
    als_p=0.01
)

correction_time = time.time() - start_time

# Convert back to DataFrames with original column names
X_train_corrected_df = pd.DataFrame(X_train_corrected, 
                                   columns=X_train_ind.columns, 
                                   index=X_train_ind.index)
X_test_corrected_df = pd.DataFrame(X_test_corrected, 
                                  columns=X_test_ind.columns, 
                                  index=X_test_ind.index)

print(f"✅ Baseline correction completed in {correction_time:.1f} seconds")
print(f"Corrected training data shape: {X_train_corrected_df.shape}")
print(f"Corrected test data shape: {X_test_corrected_df.shape}")

# Quality check: Compare statistics before and after correction
print(f"\n=== BASELINE CORRECTION QUALITY CHECK ===")
print(f"{'Metric':<15} {'Original Train':<15} {'Corrected Train':<15} {'Change':<10}")
print("-" * 60)

orig_mean = np.mean(X_train_ind.values)
corr_mean = np.mean(X_train_corrected)
print(f"{'Mean':<15} {orig_mean:<15.2f} {corr_mean:<15.2f} {corr_mean-orig_mean:<10.2f}")

orig_std = np.std(X_train_ind.values)
corr_std = np.std(X_train_corrected)
print(f"{'Std Dev':<15} {orig_std:<15.2f} {corr_std:<15.2f} {corr_std-orig_std:<10.2f}")

orig_min = np.min(X_train_ind.values)
corr_min = np.min(X_train_corrected)
print(f"{'Minimum':<15} {orig_min:<15.2f} {corr_min:<15.2f} {corr_min-orig_min:<10.2f}")

orig_max = np.max(X_train_ind.values)
corr_max = np.max(X_train_corrected)
print(f"{'Maximum':<15} {orig_max:<15.2f} {corr_max:<15.2f} {corr_max-orig_max:<10.2f}")

# Visualization of baseline correction effect
print(f"\nCreating baseline correction visualization...")

# Select a representative spectrum for visualization
sample_idx = 42  # Use a fixed index for reproducibility
sample_spectrum_orig = X_train_ind.iloc[sample_idx].values
sample_spectrum_corr = X_train_corrected[sample_idx]
wavelengths = np.array([float(col) for col in X_train_ind.columns])

# Calculate baseline
sample_baseline = baseline_hybrid_4s_als(sample_spectrum_orig, 
                                        window_length=51, polyorder=3, 
                                        als_lam=1e4, als_p=0.01)

# Create visualization
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Original spectrum with baseline
axes[0, 0].plot(wavelengths, sample_spectrum_orig, 'b-', label='Original Spectrum', linewidth=1)
axes[0, 0].plot(wavelengths, sample_baseline, 'r--', label='Hybrid 4S+AsLS Baseline', linewidth=2)
axes[0, 0].set_title('Original Spectrum with Baseline Estimate')
axes[0, 0].set_xlabel('Wavelength (nm)')
axes[0, 0].set_ylabel('Intensity')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Baseline-corrected spectrum
axes[0, 1].plot(wavelengths, sample_spectrum_orig, 'b-', alpha=0.3, label='Original')
axes[0, 1].plot(wavelengths, sample_spectrum_corr, 'g-', label='Baseline Corrected', linewidth=1.5)
axes[0, 1].set_title('Baseline-Corrected Spectrum')
axes[0, 1].set_xlabel('Wavelength (nm)')
axes[0, 1].set_ylabel('Intensity')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Distribution comparison
axes[1, 0].hist(X_train_ind.values.flatten()[::100], bins=50, alpha=0.5, label='Original', density=True)
axes[1, 0].hist(X_train_corrected.flatten()[::100], bins=50, alpha=0.5, label='Corrected', density=True)
axes[1, 0].set_title('Intensity Distribution Comparison')
axes[1, 0].set_xlabel('Intensity')
axes[1, 0].set_ylabel('Density')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Class-wise intensity statistics
class_stats_orig = []
class_stats_corr = []
classes = y_train_ind.unique()

for cls in classes:
    cls_mask = y_train_ind == cls
    orig_mean = np.mean(X_train_ind[cls_mask].values)
    corr_mean = np.mean(X_train_corrected[cls_mask])
    class_stats_orig.append(orig_mean)
    class_stats_corr.append(corr_mean)

x_pos = np.arange(len(classes))
width = 0.35

axes[1, 1].bar(x_pos - width/2, class_stats_orig, width, label='Original', alpha=0.7)
axes[1, 1].bar(x_pos + width/2, class_stats_corr, width, label='Corrected', alpha=0.7)
axes[1, 1].set_title('Mean Intensity by Class')
axes[1, 1].set_xlabel('Tire Component')
axes[1, 1].set_ylabel('Mean Intensity')
axes[1, 1].set_xticks(x_pos)
axes[1, 1].set_xticklabels(classes)
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n🎯 Ready for Random Forest analysis with baseline-corrected data!")
print(f"   • Applied: Hybrid 4S+AsLS baseline correction")
print(f"   • Processing time: {correction_time:.1f} seconds")
print(f"   • Training samples: {len(X_train_corrected_df):,}")
print(f"   • Test samples: {len(X_test_corrected_df):,}")
print(f"   • Features: {X_train_corrected_df.shape[1]}")

# Update the data variables to use baseline-corrected versions
print(f"\nUpdating data variables to use baseline-corrected spectra...")
X_train_original = X_train_corrected_df.copy()
X_test_original = X_test_corrected_df.copy()

In [12]:
%%time

# 2. SVM Analysis on Original Features with Multiple Kernels
print("=" * 60)
print("2. SVM ANALYSIS ON ORIGINAL FEATURES")
print("=" * 60)

# Define comprehensive SVM configurations
svm_configs = {
    'Linear': {
        'kernel': 'linear',
        'param_grid': {
            'C': [0.01, 0.1, 1, 10, 100, 1000]
        }
    },
    'RBF': {
        'kernel': 'rbf', 
        'param_grid': {
            'C': [0.1, 1, 10, 100, 1000],
            'gamma': ['scale', 'auto', 0.001, 0.01, 0.1, 1]
        }
    },
    'Polynomial': {
        'kernel': 'poly',
        'param_grid': {
            'C': [0.1, 1, 10, 100],
            'degree': [2, 3, 4],
            'gamma': ['scale', 'auto', 0.01, 0.1]
        }
    },
    'Sigmoid': {
        'kernel': 'sigmoid',
        'param_grid': {
            'C': [0.1, 1, 10, 100],
            'gamma': ['scale', 'auto', 0.01, 0.1, 1]
        }
    }
}

scalers = {
    'StandardScaler': StandardScaler(),
    'MinMaxScaler': MinMaxScaler(),
    'RobustScaler': RobustScaler()
}
# Cross-validation setup
cv_folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Store results for each kernel and scaler combination
original_results = {}
best_original_result = {'accuracy': 0, 'kernel': '', 'scaler': '', 'params': {}}

print("Testing SVM kernels with different scalers on original features...")
print("This may take several minutes due to high dimensionality...")

for scaler_name, scaler in scalers.items():
    print(f"\n--- Testing with {scaler_name} ---")
    
    # Scale the data
    X_train_scaled = scaler.fit_transform(X_train_ind)
    X_test_scaled = scaler.transform(X_test_ind)

    original_results[scaler_name] = {}
    
    for kernel_name, config in svm_configs.items():
        print(f"  {kernel_name} kernel...", end=" ")
        start_time = time.time()
        
        try:
            # Create SVM with specific kernel
            svm = SVC(kernel=config['kernel'], random_state=42, probability=True)
            
            # Grid search for optimal parameters
            grid_search = GridSearchCV(
                svm, 
                config['param_grid'], 
                cv=cv_folds, 
                scoring='accuracy',
                n_jobs=-1,
                verbose=0
            )
            
            # Fit and evaluate
            grid_search.fit(X_train_scaled, y_train_ind)
            
            # Test set performance
            test_accuracy = grid_search.score(X_test_scaled, y_test_ind)
            
            # Store results
            result = {
                'best_score': grid_search.best_score_,
                'test_accuracy': test_accuracy,
                'best_params': grid_search.best_params_,
                'n_support_vectors': grid_search.best_estimator_.n_support_.sum(),
                'support_vector_ratio': grid_search.best_estimator_.n_support_.sum() / len(X_train_scaled),
                'training_time': time.time() - start_time,
                'best_estimator': grid_search.best_estimator_
            }
            
            original_results[scaler_name][kernel_name] = result
            
            # Track best overall result
            if test_accuracy > best_original_result['accuracy']:
                best_original_result = {
                    'accuracy': test_accuracy,
                    'kernel': kernel_name,
                    'scaler': scaler_name,
                    'params': grid_search.best_params_,
                    'estimator': grid_search.best_estimator_
                }
            
            print(f"CV: {grid_search.best_score_:.4f}, Test: {test_accuracy:.4f} ({time.time()-start_time:.1f}s)")
            
        except Exception as e:
            print(f"Error: {str(e)}")
            original_results[scaler_name][kernel_name] = {
                'error': str(e),
                'test_accuracy': 0,
                'best_score': 0
            }

print(f"\n🏆 Best Original Features Result:")
print(f"   Kernel: {best_original_result['kernel']}")
print(f"   Scaler: {best_original_result['scaler']}")
print(f"   Test Accuracy: {best_original_result['accuracy']:.4f}")
print(f"   Parameters: {best_original_result['params']}")

# Create comprehensive results visualization
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))

# 1. Accuracy heatmap by kernel and scaler
accuracy_matrix = []
kernel_names = list(svm_configs.keys())
scaler_names = list(scalers.keys())

for scaler_name in scaler_names:
    row = []
    for kernel_name in kernel_names:
        if kernel_name in original_results[scaler_name] and 'test_accuracy' in original_results[scaler_name][kernel_name]:
            row.append(original_results[scaler_name][kernel_name]['test_accuracy'])
        else:
            row.append(0)
    accuracy_matrix.append(row)

accuracy_df = pd.DataFrame(accuracy_matrix, index=scaler_names, columns=kernel_names)
sns.heatmap(accuracy_df, annot=True, fmt='.4f', cmap='viridis', ax=ax1)
ax1.set_title('SVM Test Accuracy by Kernel and Scaler', fontsize=14, fontweight='bold')
ax1.set_xlabel('Kernel Type')
ax1.set_ylabel('Scaler Type')

# 2. Support vector ratios
sv_ratios = []
labels = []
for scaler_name in scaler_names:
    for kernel_name in kernel_names:
        if (kernel_name in original_results[scaler_name] and 
            'support_vector_ratio' in original_results[scaler_name][kernel_name]):
            sv_ratios.append(original_results[scaler_name][kernel_name]['support_vector_ratio'])
            labels.append(f"{kernel_name}\n({scaler_name})")

bars = ax2.bar(range(len(sv_ratios)), sv_ratios, alpha=0.7)
ax2.set_title('Support Vector Ratios by Configuration', fontsize=14, fontweight='bold')
ax2.set_ylabel('Support Vector Ratio')
ax2.set_xlabel('Kernel (Scaler)')
ax2.set_xticks(range(len(labels)))
ax2.set_xticklabels(labels, rotation=45, ha='right')
ax2.grid(axis='y', alpha=0.3)

# Highlight best performing bar
best_idx = np.argmax([acc for row in accuracy_matrix for acc in row])
bars[best_idx].set_color('red')
bars[best_idx].set_alpha(0.9)

# 3. Training time comparison
training_times = []
for scaler_name in scaler_names:
    for kernel_name in kernel_names:
        if (kernel_name in original_results[scaler_name] and 
            'training_time' in original_results[scaler_name][kernel_name]):
            training_times.append(original_results[scaler_name][kernel_name]['training_time'])

ax3.bar(range(len(training_times)), training_times, alpha=0.7, color='orange')
ax3.set_title('Training Time by Configuration', fontsize=14, fontweight='bold')
ax3.set_ylabel('Training Time (seconds)')
ax3.set_xlabel('Kernel (Scaler)')
ax3.set_xticks(range(len(labels)))
ax3.set_xticklabels(labels, rotation=45, ha='right')
ax3.grid(axis='y', alpha=0.3)

# 4. Accuracy vs Training Time scatter
accuracies_flat = [acc for row in accuracy_matrix for acc in row]
ax4.scatter(training_times, accuracies_flat, s=100, alpha=0.7, c=range(len(training_times)), cmap='viridis')
ax4.set_xlabel('Training Time (seconds)')
ax4.set_ylabel('Test Accuracy')
ax4.set_title('Accuracy vs Training Time Trade-off', fontsize=14, fontweight='bold')
ax4.grid(True, alpha=0.3)

# Annotate best point
best_acc_idx = np.argmax(accuracies_flat)
ax4.annotate(f'Best: {labels[best_acc_idx]}', 
            xy=(training_times[best_acc_idx], accuracies_flat[best_acc_idx]),
            xytext=(10, 10), textcoords='offset points',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='red', alpha=0.3),
            arrowprops=dict(arrowstyle='->', connectionstyle='arc3,rad=0'))

plt.tight_layout()
plt.show()

# Detailed classification report for best model
print(f"\n📊 Detailed Results for Best Model ({best_original_result['kernel']} + {best_original_result['scaler']}):")
print("=" * 80)

best_scaler = scalers[best_original_result['scaler']]
X_train_best = best_scaler.fit_transform(X_train_original)
X_test_best = best_scaler.transform(X_test_original)

y_pred = best_original_result['estimator'].predict(X_test_best)
y_pred_proba = best_original_result['estimator'].predict_proba(X_test_best)

print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=class_names))

print(f"\nConfusion Matrix:")
cm = confusion_matrix(y_test, y_pred)
print(cm)

print(f"\nModel Details:")
print(f"- Support Vectors: {best_original_result['estimator'].n_support_.sum()} ({best_original_result['estimator'].n_support_.sum()/len(X_train_best)*100:.1f}% of training data)")
print(f"- Support Vectors per class: {dict(zip(class_names, best_original_result['estimator'].n_support_))}")
print(f"- Number of classes: {len(class_names)}")

print(f"\nOriginal features SVM analysis complete!")
print(f"Best configuration: {best_original_result['kernel']} kernel with {best_original_result['scaler']}")
print(f"Achieved {best_original_result['accuracy']:.4f} accuracy on {X_train.shape[1]} features")

2. SVM ANALYSIS ON ORIGINAL FEATURES
Testing SVM kernels with different scalers on original features...
This may take several minutes due to high dimensionality...

--- Testing with StandardScaler ---
  Linear kernel... 

KeyboardInterrupt: 

In [13]:
%%time

# 3. SVM Analysis with PCA Dimensionality Reduction
print("=" * 60)
print("3. SVM ANALYSIS WITH PCA DIMENSIONALITY REDUCTION")
print("=" * 60)

# PCA configurations to test
pca_configs = [
    {'variance': 0.95, 'name': '95% Variance'},
    {'variance': 0.99, 'name': '99% Variance'},
    {'n_components': 3, 'name': '3 Components'},
    {'n_components': 28, 'name': '28 Components'},
    {'n_components': 100, 'name': '100 Components'}
]

pca_results = {}
best_pca_result = {'accuracy': 0, 'config': '', 'kernel': '', 'scaler': ''}

print("Testing SVM with PCA-reduced features...")
print("Evaluating multiple PCA configurations and kernel combinations...")

for pca_config in pca_configs:
    config_name = pca_config['name']
    print(f"\n--- PCA Configuration: {config_name} ---")
    
    # Apply PCA with current configuration
    if 'variance' in pca_config:
        pca = PCA(n_components=pca_config['variance'], random_state=42)
    else:
        pca = PCA(n_components=pca_config['n_components'], random_state=42)
    
    # Fit PCA on scaled training data (using best scaler from original analysis)
    best_scaler_name = best_original_result['scaler']
    scaler = scalers[best_scaler_name]
    
    X_train_scaled = scaler.fit_transform(X_train_original)
    X_test_scaled = scaler.transform(X_test_original)
    
    X_train_pca = pca.fit_transform(X_train_scaled)
    X_test_pca = pca.transform(X_test_scaled)
    
    actual_components = X_train_pca.shape[1]
    explained_variance = pca.explained_variance_ratio_.sum()
    
    print(f"  Actual components: {actual_components}")
    print(f"  Explained variance: {explained_variance:.4f}")
    print(f"  Dimensionality reduction: {(1 - actual_components/X_train_original.shape[1])*100:.1f}%")
    
    pca_results[config_name] = {
        'n_components': actual_components,
        'explained_variance': explained_variance,
        'kernels': {}
    }
    
    # Test different kernels on PCA-reduced data
    for kernel_name, config in svm_configs.items():
        print(f"    {kernel_name} kernel...", end=" ")
        start_time = time.time()
        
        try:
            # Create SVM
            svm = SVC(kernel=config['kernel'], random_state=42, probability=True)
            
            # Reduced parameter grid for efficiency with PCA
            reduced_param_grid = config['param_grid'].copy()
            if kernel_name == 'RBF':
                reduced_param_grid['C'] = [0.1, 1, 10, 100]
                reduced_param_grid['gamma'] = ['scale', 0.01, 0.1, 1]
            elif kernel_name == 'Polynomial':
                reduced_param_grid['C'] = [0.1, 1, 10]
                reduced_param_grid['degree'] = [2, 3]
                reduced_param_grid['gamma'] = ['scale', 0.01]
            
            # Grid search
            grid_search = GridSearchCV(
                svm, 
                reduced_param_grid, 
                cv=cv_folds, 
                scoring='accuracy',
                n_jobs=-1,
                verbose=0
            )
            
            grid_search.fit(X_train_pca, y_train)
            test_accuracy = grid_search.score(X_test_pca, y_test)
            
            result = {
                'best_score': grid_search.best_score_,
                'test_accuracy': test_accuracy,
                'best_params': grid_search.best_params_,
                'n_support_vectors': grid_search.best_estimator_.n_support_.sum(),
                'support_vector_ratio': grid_search.best_estimator_.n_support_.sum() / len(X_train_pca),
                'training_time': time.time() - start_time,
                'best_estimator': grid_search.best_estimator_
            }
            
            pca_results[config_name]['kernels'][kernel_name] = result
            
            # Track best PCA result
            if test_accuracy > best_pca_result['accuracy']:
                best_pca_result = {
                    'accuracy': test_accuracy,
                    'config': config_name,
                    'kernel': kernel_name,
                    'scaler': best_scaler_name,
                    'n_components': actual_components,
                    'explained_variance': explained_variance,
                    'params': grid_search.best_params_,
                    'estimator': grid_search.best_estimator_,
                    'pca_transformer': pca
                }
            
            print(f"CV: {grid_search.best_score_:.4f}, Test: {test_accuracy:.4f} ({time.time()-start_time:.1f}s)")
            
        except Exception as e:
            print(f"Error: {str(e)}")
            pca_results[config_name]['kernels'][kernel_name] = {'error': str(e)}

print(f"\n🏆 Best PCA Result:")
print(f"   Configuration: {best_pca_result['config']}")
print(f"   Kernel: {best_pca_result['kernel']}")
print(f"   Components: {best_pca_result['n_components']}")
print(f"   Explained Variance: {best_pca_result['explained_variance']:.4f}")
print(f"   Test Accuracy: {best_pca_result['accuracy']:.4f}")
print(f"   Parameters: {best_pca_result['params']}")

# Visualization of PCA results
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))

# 1. PCA explained variance
pca_names = []
explained_vars = []
n_components_list = []

for config_name, results in pca_results.items():
    if 'explained_variance' in results:
        pca_names.append(config_name)
        explained_vars.append(results['explained_variance'])
        n_components_list.append(results['n_components'])

bars1 = ax1.bar(pca_names, explained_vars, alpha=0.7, color='skyblue')
ax1.set_title('Explained Variance by PCA Configuration', fontsize=14, fontweight='bold')
ax1.set_ylabel('Explained Variance Ratio')
ax1.set_ylim(0, 1.1)
ax1.tick_params(axis='x', rotation=45)
ax1.grid(axis='y', alpha=0.3)

for bar, var in zip(bars1, explained_vars):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{var:.3f}', ha='center', va='bottom', fontweight='bold')

# 2. Number of components
bars2 = ax2.bar(pca_names, n_components_list, alpha=0.7, color='lightgreen')
ax2.set_title('Number of PCA Components', fontsize=14, fontweight='bold')
ax2.set_ylabel('Number of Components')
ax2.tick_params(axis='x', rotation=45)
ax2.grid(axis='y', alpha=0.3)

for bar, n_comp in zip(bars2, n_components_list):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
             f'{n_comp}', ha='center', va='bottom', fontweight='bold')

# 3. Best accuracy by PCA configuration and kernel
accuracy_data = []
config_labels = []
kernel_labels = []

for config_name, results in pca_results.items():
    if 'kernels' in results:
        for kernel_name, kernel_result in results['kernels'].items():
            if 'test_accuracy' in kernel_result:
                accuracy_data.append(kernel_result['test_accuracy'])
                config_labels.append(config_name)
                kernel_labels.append(kernel_name)

# Create grouped bar chart
unique_configs = list(pca_results.keys())
unique_kernels = list(svm_configs.keys())
x_pos = np.arange(len(unique_configs))
width = 0.2

for i, kernel in enumerate(unique_kernels):
    kernel_accuracies = []
    for config in unique_configs:
        if (config in pca_results and 'kernels' in pca_results[config] and 
            kernel in pca_results[config]['kernels'] and 
            'test_accuracy' in pca_results[config]['kernels'][kernel]):
            kernel_accuracies.append(pca_results[config]['kernels'][kernel]['test_accuracy'])
        else:
            kernel_accuracies.append(0)
    
    ax3.bar(x_pos + i*width, kernel_accuracies, width, label=kernel, alpha=0.7)

ax3.set_title('SVM Accuracy by PCA Configuration and Kernel', fontsize=14, fontweight='bold')
ax3.set_ylabel('Test Accuracy')
ax3.set_xlabel('PCA Configuration')
ax3.set_xticks(x_pos + width * 1.5)
ax3.set_xticklabels(unique_configs, rotation=45, ha='right')
ax3.legend()
ax3.grid(axis='y', alpha=0.3)

# 4. Efficiency plot: Accuracy vs Components
for config_name, results in pca_results.items():
    if 'kernels' in results:
        best_kernel_acc = 0
        for kernel_result in results['kernels'].values():
            if 'test_accuracy' in kernel_result:
                best_kernel_acc = max(best_kernel_acc, kernel_result['test_accuracy'])
        
        if best_kernel_acc > 0:
            ax4.scatter(results['n_components'], best_kernel_acc, s=100, alpha=0.7, 
                       label=config_name)

ax4.set_xlabel('Number of Components')
ax4.set_ylabel('Best Test Accuracy')
ax4.set_title('Accuracy vs Dimensionality Trade-off', fontsize=14, fontweight='bold')
ax4.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# PCA feature importance analysis
print(f"\n📊 PCA Feature Analysis for Best Configuration:")
print("=" * 60)

best_pca = best_pca_result['pca_transformer']
print(f"Top 10 most important original features (by PCA component weights):")

# Calculate feature importance from first few principal components
feature_importance = np.abs(best_pca.components_[:5]).mean(axis=0)
top_features = np.argsort(feature_importance)[-10:][::-1]

for i, feat_idx in enumerate(top_features):
    print(f"  {i+1:2d}. Feature {feat_idx:4d}: {feature_importance[feat_idx]:.4f}")

print(f"\nPCA analysis complete!")
print(f"Best: {best_pca_result['config']} + {best_pca_result['kernel']} kernel")
print(f"Reduced from {X_train_original.shape[1]} to {best_pca_result['n_components']} features")
print(f"Achieved {best_pca_result['accuracy']:.4f} accuracy with {best_pca_result['explained_variance']:.1%} variance")

3. SVM ANALYSIS WITH PCA DIMENSIONALITY REDUCTION
Testing SVM with PCA-reduced features...
Evaluating multiple PCA configurations and kernel combinations...

--- PCA Configuration: 95% Variance ---


KeyError: ''

In [14]:
%%time

# 4. SVM Analysis with LDA Dimensionality Reduction  
print("=" * 60)
print("4. SVM ANALYSIS WITH LDA DIMENSIONALITY REDUCTION")
print("=" * 60)

# LDA can reduce to at most n_classes - 1 dimensions
n_classes = len(np.unique(y_train))
max_lda_components = n_classes - 1

print(f"Number of classes: {n_classes}")
print(f"Maximum LDA components: {max_lda_components}")

# Apply LDA dimensionality reduction
lda = LinearDiscriminantAnalysis(n_components=max_lda_components)

# Use best scaler from previous analysis
best_scaler = scalers[best_original_result['scaler']]
X_train_scaled = best_scaler.fit_transform(X_train_original)
X_test_scaled = best_scaler.transform(X_test_original)

# Fit LDA and transform data
X_train_lda = lda.fit_transform(X_train_scaled, y_train)
X_test_lda = lda.transform(X_test_scaled)

print(f"LDA transformation complete:")
print(f"  Original features: {X_train_scaled.shape[1]}")
print(f"  LDA features: {X_train_lda.shape[1]}")
print(f"  Dimensionality reduction: {(1 - X_train_lda.shape[1]/X_train_scaled.shape[1])*100:.1f}%")

# Calculate explained variance ratio for LDA
lda_explained_variance = lda.explained_variance_ratio_
print(f"  Explained variance per component: {lda_explained_variance}")
print(f"  Total explained variance: {lda_explained_variance.sum():.4f}")

# Test all kernels on LDA-reduced data
lda_results = {}
best_lda_result = {'accuracy': 0, 'kernel': '', 'params': {}}

print(f"\nTesting SVM kernels on LDA-reduced features ({max_lda_components} components)...")

for kernel_name, config in svm_configs.items():
    print(f"  {kernel_name} kernel...", end=" ")
    start_time = time.time()
    
    try:
        # Create SVM
        svm = SVC(kernel=config['kernel'], random_state=42, probability=True)
        
        # Simplified parameter grid for LDA (low dimensionality)
        simplified_param_grid = config['param_grid'].copy()
        if kernel_name == 'RBF':
            simplified_param_grid['C'] = [0.1, 1, 10, 100, 1000]
            simplified_param_grid['gamma'] = ['scale', 'auto', 0.001, 0.01, 0.1, 1]
        elif kernel_name == 'Polynomial':
            simplified_param_grid['C'] = [0.1, 1, 10, 100]
            simplified_param_grid['degree'] = [2, 3, 4]
            simplified_param_grid['gamma'] = ['scale', 'auto', 0.01, 0.1]
        
        # Grid search
        grid_search = GridSearchCV(
            svm, 
            simplified_param_grid, 
            cv=cv_folds, 
            scoring='accuracy',
            n_jobs=-1,
            verbose=0
        )
        
        grid_search.fit(X_train_lda, y_train)
        test_accuracy = grid_search.score(X_test_lda, y_test)
        
        result = {
            'best_score': grid_search.best_score_,
            'test_accuracy': test_accuracy,
            'best_params': grid_search.best_params_,
            'n_support_vectors': grid_search.best_estimator_.n_support_.sum(),
            'support_vector_ratio': grid_search.best_estimator_.n_support_.sum() / len(X_train_lda),
            'training_time': time.time() - start_time,
            'best_estimator': grid_search.best_estimator_
        }
        
        lda_results[kernel_name] = result
        
        # Track best LDA result
        if test_accuracy > best_lda_result['accuracy']:
            best_lda_result = {
                'accuracy': test_accuracy,
                'kernel': kernel_name,
                'params': grid_search.best_params_,
                'estimator': grid_search.best_estimator_,
                'n_components': max_lda_components
            }
        
        print(f"CV: {grid_search.best_score_:.4f}, Test: {test_accuracy:.4f} ({time.time()-start_time:.1f}s)")
        
    except Exception as e:
        print(f"Error: {str(e)}")
        lda_results[kernel_name] = {'error': str(e)}

print(f"\n🏆 Best LDA Result:")
print(f"   Kernel: {best_lda_result['kernel']}")
print(f"   Components: {best_lda_result['n_components']}")
print(f"   Test Accuracy: {best_lda_result['accuracy']:.4f}")
print(f"   Parameters: {best_lda_result['params']}")

# Visualization of LDA results
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))

# 1. LDA explained variance per component
components = [f'LD{i+1}' for i in range(len(lda_explained_variance))]
bars1 = ax1.bar(components, lda_explained_variance, alpha=0.7, color='lightblue')
ax1.set_title('LDA Explained Variance by Component', fontsize=14, fontweight='bold')
ax1.set_ylabel('Explained Variance Ratio')
ax1.set_xlabel('Linear Discriminant')
ax1.grid(axis='y', alpha=0.3)

for bar, var in zip(bars1, lda_explained_variance):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{var:.3f}', ha='center', va='bottom', fontweight='bold')

# 2. Kernel performance comparison on LDA features
kernel_names = [k for k in lda_results.keys() if 'error' not in lda_results[k]]
kernel_accuracies = [lda_results[k]['test_accuracy'] for k in kernel_names]
kernel_times = [lda_results[k]['training_time'] for k in kernel_names]

bars2 = ax2.bar(kernel_names, kernel_accuracies, alpha=0.7, color='lightgreen')
ax2.set_title('SVM Kernel Performance on LDA Features', fontsize=14, fontweight='bold')
ax2.set_ylabel('Test Accuracy')
ax2.set_xlabel('Kernel Type')
ax2.set_ylim(0, 1.1)
ax2.grid(axis='y', alpha=0.3)

for bar, acc in zip(bars2, kernel_accuracies):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{acc:.4f}', ha='center', va='bottom', fontweight='bold')

# Highlight best kernel
best_kernel_idx = np.argmax(kernel_accuracies)
bars2[best_kernel_idx].set_color('red')
bars2[best_kernel_idx].set_alpha(0.9)

# 3. Support vector analysis
sv_counts = [lda_results[k]['n_support_vectors'] for k in kernel_names]
sv_ratios = [lda_results[k]['support_vector_ratio'] for k in kernel_names]

bars3 = ax3.bar(kernel_names, sv_ratios, alpha=0.7, color='orange')
ax3.set_title('Support Vector Ratios on LDA Features', fontsize=14, fontweight='bold')
ax3.set_ylabel('Support Vector Ratio')
ax3.set_xlabel('Kernel Type')
ax3.grid(axis='y', alpha=0.3)

for bar, ratio in zip(bars3, sv_ratios):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
             f'{ratio:.3f}', ha='center', va='bottom', fontweight='bold')

# 4. LDA 2D visualization (if we have 2+ components)
if max_lda_components >= 2:
    # Create scatter plot of first two LDA components
    scatter = ax4.scatter(X_train_lda[:, 0], X_train_lda[:, 1], c=y_train, alpha=0.6, cmap='viridis')
    ax4.set_xlabel(f'LD1 (Explained Var: {lda_explained_variance[0]:.3f})')
    ax4.set_ylabel(f'LD2 (Explained Var: {lda_explained_variance[1]:.3f})')
    ax4.set_title('LDA Feature Space Visualization', fontsize=14, fontweight='bold')
    
    # Add class labels
    for i, class_name in enumerate(class_names):
        mask = y_train == i
        ax4.scatter(X_train_lda[mask, 0], X_train_lda[mask, 1], 
                   label=class_name, alpha=0.6, s=30)
    
    ax4.legend()
    ax4.grid(True, alpha=0.3)
else:
    # Single component case
    ax4.hist([X_train_lda[y_train == i, 0] for i in range(n_classes)], 
             alpha=0.7, label=class_names, bins=30)
    ax4.set_xlabel('LD1')
    ax4.set_ylabel('Frequency')
    ax4.set_title('LDA Single Component Distribution', fontsize=14, fontweight='bold')
    ax4.legend()
    ax4.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

# Detailed analysis of best LDA model
print(f"\n📊 Detailed LDA Analysis:")
print("=" * 50)

# Get predictions from best LDA model
y_pred_lda = best_lda_result['estimator'].predict(X_test_lda)
y_pred_proba_lda = best_lda_result['estimator'].predict_proba(X_test_lda)

print("Classification Report for Best LDA Model:")
print(classification_report(y_test, y_pred_lda, target_names=class_names))

print(f"\nConfusion Matrix:")
cm_lda = confusion_matrix(y_test, y_pred_lda)
print(cm_lda)

# LDA discriminant analysis
print(f"\nLDA Discriminant Analysis:")
print(f"- Linear discriminants: {max_lda_components}")
print(f"- Total separation power: {lda_explained_variance.sum():.4f}")
print(f"- Most discriminative LD: LD{np.argmax(lda_explained_variance)+1} ({lda_explained_variance.max():.4f})")

# Compare efficiency
print(f"\n💡 LDA Efficiency Analysis:")
print(f"- Features reduced from {X_train_original.shape[1]} to {max_lda_components}")
print(f"- Dimensionality reduction: {(1 - max_lda_components/X_train_original.shape[1])*100:.1f}%")
print(f"- Best accuracy: {best_lda_result['accuracy']:.4f}")
print(f"- Support vectors: {best_lda_result['estimator'].n_support_.sum()} ({best_lda_result['estimator'].n_support_.sum()/len(X_train_lda)*100:.1f}%)")

print(f"\nLDA analysis complete!")
print(f"Best: {best_lda_result['kernel']} kernel on {max_lda_components} LDA components")
print(f"Achieved {best_lda_result['accuracy']:.4f} accuracy with maximum class separability")

4. SVM ANALYSIS WITH LDA DIMENSIONALITY REDUCTION


NameError: name 'y_train' is not defined

In [15]:
%%time

# 5. SVM Interpretability and Support Vector Analysis
print("=" * 60)
print("5. SVM INTERPRETABILITY AND SUPPORT VECTOR ANALYSIS")
print("=" * 60)

# Analyze the best-performing SVM models from each configuration
models_to_analyze = {
    'Original Features': {
        'model': best_original_result['estimator'],
        'X_train': best_scaler.fit_transform(X_train_original),
        'X_test': best_scaler.transform(X_test_original),
        'kernel': best_original_result['kernel'],
        'features': X_train_original.shape[1],
        'config': f"{best_original_result['kernel']} + {best_original_result['scaler']}"
    },
    'PCA Features': {
        'model': best_pca_result['estimator'],
        'X_train': best_pca_result['pca_transformer'].transform(best_scaler.fit_transform(X_train_original)),
        'X_test': best_pca_result['pca_transformer'].transform(best_scaler.transform(X_test_original)),
        'kernel': best_pca_result['kernel'],
        'features': best_pca_result['n_components'],
        'config': f"{best_pca_result['kernel']} + PCA({best_pca_result['n_components']})"
    },
    'LDA Features': {
        'model': best_lda_result['estimator'],
        'X_train': X_train_lda,
        'X_test': X_test_lda,
        'kernel': best_lda_result['kernel'],
        'features': max_lda_components,
        'config': f"{best_lda_result['kernel']} + LDA({max_lda_components})"
    }
}

print("Analyzing support vector patterns and model interpretability...")

# Support vector analysis
support_vector_analysis = {}

for config_name, config in models_to_analyze.items():
    model = config['model']
    X_train_transformed = config['X_train']
    
    print(f"\n--- {config_name} Analysis ---")
    print(f"Configuration: {config['config']}")
    print(f"Kernel: {config['kernel']}")
    print(f"Features: {config['features']}")
    
    # Support vector statistics
    n_sv_total = model.n_support_.sum()
    n_sv_per_class = model.n_support_
    sv_ratio = n_sv_total / len(X_train_transformed)
    
    print(f"Support Vectors:")
    print(f"  Total: {n_sv_total} ({sv_ratio:.3f} of training data)")
    print(f"  Per class: {dict(zip(class_names, n_sv_per_class))}")
    
    # Decision function analysis
    decision_values = model.decision_function(config['X_test'])
    if len(class_names) == 2:
        # Binary classification
        margin_distances = np.abs(decision_values)
        confidence_scores = model.predict_proba(config['X_test']).max(axis=1)
    else:
        # Multi-class classification (one-vs-one)
        margin_distances = np.abs(decision_values).mean(axis=1)
        confidence_scores = model.predict_proba(config['X_test']).max(axis=1)
    
    print(f"Decision Analysis:")
    print(f"  Average margin distance: {margin_distances.mean():.4f}")
    print(f"  Margin std: {margin_distances.std():.4f}")
    print(f"  Average confidence: {confidence_scores.mean():.4f}")
    print(f"  Confidence std: {confidence_scores.std():.4f}")
    
    # Store for comparison
    support_vector_analysis[config_name] = {
        'n_support_vectors': n_sv_total,
        'support_vector_ratio': sv_ratio,
        'n_support_per_class': n_sv_per_class,
        'margin_distances': margin_distances,
        'confidence_scores': confidence_scores,
        'kernel': config['kernel'],
        'features': config['features']
    }

# Comprehensive visualization of SVM interpretability
fig = plt.figure(figsize=(20, 15))

# Create a 3x3 grid for visualizations
gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)

# 1. Support vector comparison
ax1 = fig.add_subplot(gs[0, 0])
configs = list(support_vector_analysis.keys())
sv_counts = [support_vector_analysis[config]['n_support_vectors'] for config in configs]
sv_ratios = [support_vector_analysis[config]['support_vector_ratio'] for config in configs]

bars = ax1.bar(configs, sv_counts, alpha=0.7, color=['skyblue', 'lightgreen', 'orange'])
ax1.set_title('Support Vector Counts', fontsize=12, fontweight='bold')
ax1.set_ylabel('Number of Support Vectors')
ax1.tick_params(axis='x', rotation=45)

for bar, count in zip(bars, sv_counts):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
             f'{count}', ha='center', va='bottom', fontweight='bold')

# 2. Support vector ratios
ax2 = fig.add_subplot(gs[0, 1])
bars = ax2.bar(configs, sv_ratios, alpha=0.7, color=['skyblue', 'lightgreen', 'orange'])
ax2.set_title('Support Vector Ratios', fontsize=12, fontweight='bold')
ax2.set_ylabel('Ratio of Training Data')
ax2.tick_params(axis='x', rotation=45)

for bar, ratio in zip(bars, sv_ratios):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{ratio:.3f}', ha='center', va='bottom', fontweight='bold')

# 3. Support vectors per class (stacked bar)
ax3 = fig.add_subplot(gs[0, 2])
class_sv_data = np.array([support_vector_analysis[config]['n_support_per_class'] 
                         for config in configs])

bottom = np.zeros(len(configs))
colors = ['red', 'green', 'blue']
for i, class_name in enumerate(class_names):
    ax3.bar(configs, class_sv_data[:, i], bottom=bottom, 
           label=class_name, alpha=0.7, color=colors[i])
    bottom += class_sv_data[:, i]

ax3.set_title('Support Vectors by Class', fontsize=12, fontweight='bold')
ax3.set_ylabel('Number of Support Vectors')
ax3.tick_params(axis='x', rotation=45)
ax3.legend()

# 4-6. Margin distance distributions
for i, config in enumerate(configs):
    ax = fig.add_subplot(gs[1, i])
    margins = support_vector_analysis[config]['margin_distances']
    
    ax.hist(margins, bins=30, alpha=0.7, color=['skyblue', 'lightgreen', 'orange'][i])
    ax.axvline(margins.mean(), color='red', linestyle='--', 
              label=f'Mean: {margins.mean():.3f}')
    ax.set_title(f'{config}\nMargin Distances', fontsize=11, fontweight='bold')
    ax.set_xlabel('Distance from Decision Boundary')
    ax.set_ylabel('Frequency')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)

# 7-9. Confidence score distributions
for i, config in enumerate(configs):
    ax = fig.add_subplot(gs[2, i])
    confidences = support_vector_analysis[config]['confidence_scores']
    
    ax.hist(confidences, bins=30, alpha=0.7, color=['skyblue', 'lightgreen', 'orange'][i])
    ax.axvline(confidences.mean(), color='red', linestyle='--', 
              label=f'Mean: {confidences.mean():.3f}')
    ax.set_title(f'{config}\nPrediction Confidence', fontsize=11, fontweight='bold')
    ax.set_xlabel('Confidence Score')
    ax.set_ylabel('Frequency')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

# Kernel-specific analysis
print(f"\n🔍 KERNEL-SPECIFIC INTERPRETABILITY ANALYSIS:")
print("=" * 60)

for config_name, analysis in support_vector_analysis.items():
    kernel = analysis['kernel']
    print(f"\n{config_name} ({kernel} kernel):")
    
    if kernel == 'linear':
        print("  ✓ Linear kernel provides direct feature interpretability")
        print("  ✓ Decision boundary is a hyperplane in original space")
        print("  ✓ Feature weights can be extracted from model coefficients")
        print(f"  ✓ Uses {analysis['support_vector_ratio']:.1%} of training data as support vectors")
        
    elif kernel == 'rbf':
        print("  ✓ RBF kernel captures non-linear patterns")
        print("  ✓ Creates complex decision boundaries in high-dimensional space")
        print("  ✓ Support vectors define local decision regions")
        print(f"  ✓ Uses {analysis['support_vector_ratio']:.1%} of training data as support vectors")
        print("  ⚠ Less interpretable than linear kernel")
        
    elif kernel == 'poly':
        print("  ✓ Polynomial kernel captures polynomial interactions")
        print("  ✓ Decision boundary complexity depends on polynomial degree")
        print("  ✓ Can model feature interactions up to specified degree")
        print(f"  ✓ Uses {analysis['support_vector_ratio']:.1%} of training data as support vectors")
        print("  ⚠ Moderate interpretability")
        
    elif kernel == 'sigmoid':
        print("  ✓ Sigmoid kernel behaves like neural network activation")
        print("  ✓ Can approximate multi-layer perceptron behavior")
        print(f"  ✓ Uses {analysis['support_vector_ratio']:.1%} of training data as support vectors")
        print("  ⚠ Limited interpretability")

# Feature importance analysis for linear kernels
linear_models = {name: analysis for name, analysis in support_vector_analysis.items() 
                if analysis['kernel'] == 'linear'}

if linear_models:
    print(f"\n📊 LINEAR KERNEL FEATURE IMPORTANCE:")
    print("=" * 50)
    
    for config_name, analysis in linear_models.items():
        print(f"\n{config_name}:")
        
        # Get the corresponding model and data
        model = models_to_analyze[config_name]['model']
        
        if hasattr(model, 'coef_'):
            if len(class_names) == 2:
                # Binary classification
                feature_importance = np.abs(model.coef_[0])
            else:
                # Multi-class: average absolute coefficients across all classes
                feature_importance = np.abs(model.coef_).mean(axis=0)
            
            # Get top features
            top_indices = np.argsort(feature_importance)[-10:][::-1]
            
            print(f"  Top 10 most important features:")
            for i, idx in enumerate(top_indices):
                print(f"    {i+1:2d}. Feature {idx:4d}: {feature_importance[idx]:.4f}")

# Model complexity analysis
print(f"\n🎯 MODEL COMPLEXITY ANALYSIS:")
print("=" * 40)

complexity_scores = {}
for config_name, analysis in support_vector_analysis.items():
    # Complexity score based on support vector ratio and feature count
    sv_complexity = analysis['support_vector_ratio'] * 100  # More SVs = more complex
    feature_complexity = np.log10(analysis['features']) * 10  # More features = more complex
    
    # Kernel complexity multiplier
    kernel_multiplier = {
        'linear': 1.0,
        'rbf': 1.5,
        'poly': 1.3,
        'sigmoid': 1.4
    }
    
    total_complexity = (sv_complexity + feature_complexity) * kernel_multiplier.get(analysis['kernel'], 1.0)
    complexity_scores[config_name] = total_complexity
    
    print(f"{config_name}:")
    print(f"  Support Vector Complexity: {sv_complexity:.1f}")
    print(f"  Feature Complexity: {feature_complexity:.1f}")
    print(f"  Kernel Multiplier: {kernel_multiplier.get(analysis['kernel'], 1.0):.1f}")
    print(f"  Total Complexity Score: {total_complexity:.1f}")

# Efficiency vs Accuracy trade-off
print(f"\n⚖️ EFFICIENCY vs ACCURACY TRADE-OFF:")
print("=" * 45)

accuracies = {
    'Original Features': best_original_result['accuracy'],
    'PCA Features': best_pca_result['accuracy'], 
    'LDA Features': best_lda_result['accuracy']
}

for config_name in configs:
    accuracy = accuracies[config_name]
    complexity = complexity_scores[config_name]
    efficiency = accuracy / (complexity / 100)  # Higher is better
    
    print(f"{config_name}:")
    print(f"  Accuracy: {accuracy:.4f}")
    print(f"  Complexity: {complexity:.1f}")
    print(f"  Efficiency Score: {efficiency:.4f}")
    print(f"  Recommendation: {'High efficiency' if efficiency > 0.02 else 'Standard efficiency'}")

print(f"\nSVM interpretability analysis complete!")
print(f"📈 Support vector patterns reveal model decision-making process")
print(f"🔍 Kernel choice significantly impacts interpretability and complexity")
print(f"⚖️ Trade-off between model accuracy and interpretability identified")

5. SVM INTERPRETABILITY AND SUPPORT VECTOR ANALYSIS


KeyError: 'estimator'

In [16]:
%%time

# 6. Comprehensive SVM Results Comparison and Analysis
print("=" * 60)
print("6. COMPREHENSIVE SVM RESULTS COMPARISON")
print("=" * 60)

# Compile all results for comprehensive comparison
all_svm_results = {
    f'{best_original_result["kernel"]} (Original)': {
        'accuracy': best_original_result['accuracy'],
        'kernel': best_original_result['kernel'],
        'features': X_train_original.shape[1],
        'scaler': best_original_result['scaler'],
        'params': best_original_result['params'],
        'dimensionality_reduction': 'None',
        'support_vectors': best_original_result['estimator'].n_support_.sum(),
        'sv_ratio': best_original_result['estimator'].n_support_.sum() / len(X_train_original),
        'preprocessing': f"{best_original_result['scaler']}"
    },
    f'{best_pca_result["kernel"]} (PCA {best_pca_result["n_components"]})': {
        'accuracy': best_pca_result['accuracy'],
        'kernel': best_pca_result['kernel'],
        'features': best_pca_result['n_components'],
        'scaler': best_pca_result['scaler'],
        'params': best_pca_result['params'],
        'dimensionality_reduction': f'{(1 - best_pca_result["n_components"]/X_train_original.shape[1])*100:.1f}%',
        'support_vectors': best_pca_result['estimator'].n_support_.sum(),
        'sv_ratio': best_pca_result['estimator'].n_support_.sum() / len(X_train_original),
        'preprocessing': f"PCA + {best_pca_result['scaler']}"
    },
    f'{best_lda_result["kernel"]} (LDA {best_lda_result["n_components"]})': {
        'accuracy': best_lda_result['accuracy'],
        'kernel': best_lda_result['kernel'],
        'features': best_lda_result['n_components'],
        'scaler': best_original_result['scaler'],  # LDA used same scaler as original
        'params': best_lda_result['params'],
        'dimensionality_reduction': f'{(1 - best_lda_result["n_components"]/X_train_original.shape[1])*100:.1f}%',
        'support_vectors': best_lda_result['estimator'].n_support_.sum(),
        'sv_ratio': best_lda_result['estimator'].n_support_.sum() / len(X_train_lda),
        'preprocessing': f"LDA + {best_original_result['scaler']}"
    }
}

# Create comprehensive summary DataFrame
summary_data = {
    'Method': list(all_svm_results.keys()),
    'Accuracy': [all_svm_results[method]['accuracy'] for method in all_svm_results.keys()],
    'Kernel': [all_svm_results[method]['kernel'] for method in all_svm_results.keys()],
    'Features': [all_svm_results[method]['features'] for method in all_svm_results.keys()],
    'Support_Vectors': [all_svm_results[method]['support_vectors'] for method in all_svm_results.keys()],
    'SV_Ratio': [f"{all_svm_results[method]['sv_ratio']:.3f}" for method in all_svm_results.keys()],
    'Dimensionality_Reduction': [all_svm_results[method]['dimensionality_reduction'] for method in all_svm_results.keys()],
    'Preprocessing': [all_svm_results[method]['preprocessing'] for method in all_svm_results.keys()]
}

results_df = pd.DataFrame(summary_data)
print("SVM COMPREHENSIVE SUMMARY TABLE:")
print("=" * 120)
print(results_df.to_string(index=False))
print("=" * 120)

# Advanced visualization dashboard
fig = plt.figure(figsize=(20, 16))
gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)

methods = results_df['Method']
accuracies = results_df['Accuracy']
features = results_df['Features']
kernels = results_df['Kernel']
sv_counts = results_df['Support_Vectors']
colors = ['skyblue', 'lightgreen', 'orange']

# 1. Accuracy comparison
ax1 = fig.add_subplot(gs[0, 0])
bars1 = ax1.bar(range(len(methods)), accuracies, color=colors, alpha=0.7)
ax1.set_title('SVM Classification Accuracy', fontsize=14, fontweight='bold')
ax1.set_ylabel('Accuracy')
ax1.set_xlabel('Method')
ax1.set_ylim(0, 1.1)
ax1.grid(axis='y', alpha=0.3)
ax1.set_xticks(range(len(methods)))
ax1.set_xticklabels([m.split(' (')[0] for m in methods], rotation=45, ha='right')

for bar, accuracy in zip(bars1, accuracies):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{accuracy:.4f}', ha='center', va='bottom', fontweight='bold')

# 2. Feature count comparison (log scale)
ax2 = fig.add_subplot(gs[0, 1])
bars2 = ax2.bar(range(len(methods)), features, color=colors, alpha=0.7)
ax2.set_title('Number of Features Used', fontsize=14, fontweight='bold')
ax2.set_ylabel('Number of Features')
ax2.set_xlabel('Method')
ax2.set_yscale('log')
ax2.grid(axis='y', alpha=0.3)
ax2.set_xticks(range(len(methods)))
ax2.set_xticklabels([m.split(' (')[0] for m in methods], rotation=45, ha='right')

for bar, feature_count in zip(bars2, features):
    ax2.text(bar.get_x() + bar.get_width()/2, feature_count * 1.5,
             f'{feature_count}', ha='center', va='bottom', fontweight='bold')

# 3. Support vector counts
ax3 = fig.add_subplot(gs[0, 2])
bars3 = ax3.bar(range(len(methods)), sv_counts, color=colors, alpha=0.7)
ax3.set_title('Support Vector Counts', fontsize=14, fontweight='bold')
ax3.set_ylabel('Number of Support Vectors')
ax3.set_xlabel('Method')
ax3.grid(axis='y', alpha=0.3)
ax3.set_xticks(range(len(methods)))
ax3.set_xticklabels([m.split(' (')[0] for m in methods], rotation=45, ha='right')

for bar, sv_count in zip(bars3, sv_counts):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
             f'{sv_count}', ha='center', va='bottom', fontweight='bold')

# 4. Kernel distribution
ax4 = fig.add_subplot(gs[1, 0])
kernel_counts = results_df['Kernel'].value_counts()
wedges, texts, autotexts = ax4.pie(kernel_counts.values, labels=kernel_counts.index, 
                                  autopct='%1.0f%%', colors=['lightblue', 'lightcoral', 'lightgreen'])
ax4.set_title('Distribution of Best Kernels', fontsize=14, fontweight='bold')

# 5. Efficiency scatter plot (Accuracy vs Features)
ax5 = fig.add_subplot(gs[1, 1])
for i, (method, accuracy, feature_count) in enumerate(zip(methods, accuracies, features)):
    ax5.scatter(feature_count, accuracy, s=200, alpha=0.7, color=colors[i], 
               edgecolors='black', linewidth=2)
    ax5.annotate(f'{i+1}', (feature_count, accuracy), 
                ha='center', va='center', fontweight='bold', color='white')

ax5.set_xlabel('Number of Features')
ax5.set_ylabel('Accuracy')
ax5.set_title('Accuracy vs Feature Efficiency', fontsize=14, fontweight='bold')
ax5.set_xscale('log')
ax5.grid(True, alpha=0.3)

# Add efficiency frontier line
efficiency_order = np.argsort(features)
ax5.plot(np.array(features)[efficiency_order], np.array(accuracies)[efficiency_order], 
         'r--', alpha=0.5, label='Efficiency Frontier')
ax5.legend()

# 6. Support vector efficiency
ax6 = fig.add_subplot(gs[1, 2])
sv_ratios_float = [float(ratio) for ratio in results_df['SV_Ratio']]
bars6 = ax6.bar(range(len(methods)), sv_ratios_float, color=colors, alpha=0.7)
ax6.set_title('Support Vector Ratios', fontsize=14, fontweight='bold')
ax6.set_ylabel('Support Vector Ratio')
ax6.set_xlabel('Method')
ax6.grid(axis='y', alpha=0.3)
ax6.set_xticks(range(len(methods)))
ax6.set_xticklabels([m.split(' (')[0] for m in methods], rotation=45, ha='right')

for bar, sv_ratio in zip(bars6, sv_ratios_float):
    ax6.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{sv_ratio:.3f}', ha='center', va='bottom', fontweight='bold')

# 7-9. Performance comparison matrices
performance_metrics = ['Accuracy', 'Efficiency', 'Interpretability']
metric_scores = {
    'Original Features': [accuracies[0], 0.3, 0.9 if kernels[0] == 'linear' else 0.3],
    'PCA Features': [accuracies[1], 0.7, 0.7],
    'LDA Features': [accuracies[2], 0.9, 0.8]
}

for i, metric in enumerate(performance_metrics):
    ax = fig.add_subplot(gs[2, i])
    
    if metric == 'Accuracy':
        scores = list(accuracies)
        ylabel = 'Accuracy Score'
    elif metric == 'Efficiency':
        # Efficiency based on features and support vectors
        scores = [1 - feat/max(features) + 1 - sv/max(sv_counts) for feat, sv in zip(features, sv_counts)]
        scores = [s/2 for s in scores]  # Normalize
        ylabel = 'Efficiency Score'
    else:  # Interpretability
        # Interpretability based on kernel type and dimensionality
        interpretability_map = {'linear': 0.9, 'rbf': 0.3, 'poly': 0.5, 'sigmoid': 0.4}
        scores = [interpretability_map.get(kernel, 0.5) + (1 - feat/max(features))*0.3 
                 for kernel, feat in zip(kernels, features)]
        ylabel = 'Interpretability Score'
    
    bars = ax.bar(range(len(methods)), scores, color=colors, alpha=0.7)
    ax.set_title(f'{metric} Comparison', fontsize=12, fontweight='bold')
    ax.set_ylabel(ylabel)
    ax.set_xlabel('Method')
    ax.set_xticks(range(len(methods)))
    ax.set_xticklabels([f'{i+1}' for i in range(len(methods))])
    ax.grid(axis='y', alpha=0.3)
    
    for bar, score in zip(bars, scores):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
               f'{score:.3f}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

# Advanced analysis and insights
print("\n🎯 SVM COMPREHENSIVE ANALYSIS INSIGHTS:")
print("=" * 50)

best_accuracy_idx = np.argmax(accuracies)
best_method = methods[best_accuracy_idx]
best_accuracy = accuracies[best_accuracy_idx]

print(f"🏆 Best performing method: {best_method}")
print(f"   └─ Accuracy: {best_accuracy:.4f}")
print(f"   └─ Kernel: {results_df.iloc[best_accuracy_idx]['Kernel']}")
print(f"   └─ Features: {results_df.iloc[best_accuracy_idx]['Features']}")
print(f"   └─ Support Vectors: {results_df.iloc[best_accuracy_idx]['Support_Vectors']}")
print(f"   └─ Preprocessing: {results_df.iloc[best_accuracy_idx]['Preprocessing']}")

print(f"\n📊 Performance Analysis:")
original_features = X_train_original.shape[1]
for i, method in enumerate(methods):
    feature_count = results_df.iloc[i]['Features']
    accuracy = accuracies.iloc[i]
    kernel = results_df.iloc[i]['Kernel']
    sv_count = results_df.iloc[i]['Support_Vectors']
    reduction = results_df.iloc[i]['Dimensionality_Reduction']
    
    print(f"   {method}:")
    print(f"   └─ Kernel: {kernel}")
    print(f"   └─ Features: {feature_count} ({reduction} reduction)")
    print(f"   └─ Accuracy: {accuracy:.4f}")
    print(f"   └─ Support Vectors: {sv_count}")
    if i > 0:
        accuracy_change = (accuracy - accuracies.iloc[0]) * 100
        print(f"   └─ Accuracy change from baseline: {accuracy_change:+.2f}%")

# Calculate comprehensive efficiency scores
efficiency_scores = []
for i, method in enumerate(methods):
    # Multi-factor efficiency: accuracy, feature reduction, SV efficiency
    norm_accuracy = accuracies[i] / max(accuracies)
    norm_features = 1 - (features[i] / max(features))  # Higher is better (fewer features)
    norm_sv = 1 - (sv_counts[i] / max(sv_counts))  # Higher is better (fewer SVs)
    
    # Weighted efficiency score
    efficiency_score = (norm_accuracy * 0.5 + norm_features * 0.3 + norm_sv * 0.2)
    efficiency_scores.append(efficiency_score)

most_efficient_idx = np.argmax(efficiency_scores)
most_efficient = methods[most_efficient_idx]

print(f"\n💡 Key Findings:")
print(f"   • Most efficient method: {most_efficient}")
print(f"     └─ Balances accuracy ({accuracies[most_efficient_idx]:.4f}) with computational efficiency")
print(f"     └─ Uses {features[most_efficient_idx]} features vs {original_features} original")
print(f"     └─ Requires {sv_counts[most_efficient_idx]} support vectors")

# Kernel analysis
kernel_performance = {}
for i, kernel in enumerate(kernels):
    if kernel not in kernel_performance:
        kernel_performance[kernel] = []
    kernel_performance[kernel].append(accuracies[i])

print(f"\n   • Kernel Analysis:")
for kernel, accs in kernel_performance.items():
    avg_acc = np.mean(accs)
    print(f"     {kernel}: {avg_acc:.4f} average accuracy")

print(f"\n   • SVM shows {'excellent' if min(accuracies) > 0.90 else 'good'} performance across configurations")
print(f"   • Dimensionality reduction significantly improves computational efficiency")
print(f"   • Support vector patterns reveal decision complexity")
print(f"   • Kernel choice strongly impacts both accuracy and interpretability")

print(f"\n🎯 SVM Recommendations:")
if accuracies[most_efficient_idx] == max(accuracies):
    print(f"   {most_efficient} provides optimal balance of accuracy and efficiency.")
elif max(accuracies) - accuracies[most_efficient_idx] < 0.02:
    print(f"   {most_efficient} offers excellent efficiency with minimal accuracy trade-off.")
    print(f"   Consider this for production deployment with computational constraints.")
else:
    print(f"   {best_method} provides highest accuracy for maximum performance.")
    print(f"   {most_efficient} offers best efficiency for resource-constrained environments.")
    print(f"   Choose based on your priority: maximum accuracy vs. computational efficiency.")

# Application-specific recommendations
print(f"\n🎯 Application-Specific Guidance:")
print(f"   • For maximum accuracy: Use {best_method}")
print(f"   • For real-time applications: Use {most_efficient}")
print(f"   • For interpretability: Use Linear kernel configurations")
print(f"   • For complex patterns: Use RBF kernel with appropriate regularization")

print(f"\n   • SVM excels at high-dimensional spectroscopy data")
print(f"   • Support vector framework provides robust decision boundaries")
print(f"   • Kernel trick enables non-linear classification without explicit feature mapping")
print(f"   • Regularization through C parameter prevents overfitting")

print(f"\n✅ SVM comprehensive analysis complete!")
print(f"   🎯 All kernel types evaluated across multiple feature representations")
print(f"   📊 Support vector patterns analyzed for interpretability")
print(f"   ⚖️ Efficiency vs accuracy trade-offs quantified")
print(f"   🔧 Ready for deployment with optimized configurations")

6. COMPREHENSIVE SVM RESULTS COMPARISON


NameError: name 'X_train_original' is not defined